# Part 2 - Analysis and Visualization

This notebook reuses the business definitions from `PART_1.ipynb`, but rebuilds the analysis from the local parquet extracts so it can run without BigQuery credentials.

Selected required visuals:
- New vs returning revenue mix by month
- Monthly churn rate vs revenue
- Customer segment split

I also add a compact experiment-design appendix for the free-shipping hypothesis from Task D.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.float_format = '{:,.2f}'.format

PALETTE = {
    'ink': '#1f2933',
    'blue': '#457b9d',
    'teal': '#2a9d8f',
    'gold': '#f4a261',
    'coral': '#e76f51',
    'sand': '#f7f5ef',
    'grid': '#d9e2ec',
}

SEGMENT_COLORS = {
    'Dormant': '#b8c0cc',
    'At risk': '#e76f51',
    'Newly acquired': '#f4a261',
    'Active repeat': '#2a9d8f',
    'Champions': '#1d3557',
}


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, start.parent]:
        if (candidate / 'thelook_parquet_data').exists():
            return candidate
    raise FileNotFoundError('Could not find thelook_parquet_data from the notebook location.')


def month_start(series: pd.Series) -> pd.Series:
    return series.dt.to_period('M').dt.to_timestamp()


def style_figure(fig: go.Figure, title: str, height: int = 520) -> go.Figure:
    fig.update_layout(
        title=title,
        template='plotly_white',
        paper_bgcolor=PALETTE['sand'],
        plot_bgcolor='white',
        font=dict(family='Avenir Next, Arial, sans-serif', color=PALETTE['ink']),
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        margin=dict(l=60, r=40, t=100, b=60),
        height=height,
    )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(gridcolor=PALETTE['grid'])
    return fig


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / 'thelook_parquet_data'

ORDER_ITEM_COLUMNS = ['id', 'order_id', 'user_id', 'status', 'created_at', 'returned_at', 'sale_price']
EVENT_COLUMNS = ['user_id', 'created_at', 'event_type']

order_items = pd.read_parquet(DATA_DIR / 'order_items.parquet', columns=ORDER_ITEM_COLUMNS)
events = pd.read_parquet(DATA_DIR / 'events.parquet', columns=EVENT_COLUMNS)

order_items['created_at'] = pd.to_datetime(order_items['created_at'], utc=True)
order_items['returned_at'] = pd.to_datetime(order_items['returned_at'], utc=True)
events['created_at'] = pd.to_datetime(events['created_at'], utc=True)

print(f'Loaded data from {DATA_DIR}')
print(f'Order items: {len(order_items):,}')
print(f'Events: {len(events):,}')


Loaded data from /Users/roger/Documents/thelook_ecommerce/thelook_parquet_data
Order items: 182,023
Events: 2,433,345


## Rebuild the Part 1 logic in pandas

The notebook keeps the same analytical definitions used in `PART_1`:
- Completed revenue excludes returned items.
- New customers are users whose first completed order lands in that month.
- Returning customers are active in a later month after their first purchase month.
- 90-day churn flags users with no future completed order in the next 90 days and censors months that have not fully matured.

Task A metrics are recreated for completeness, but the visual focus stays on mix, churn, and segmentation.


In [3]:
completed_items = order_items[(order_items['status'] == 'Complete') & (order_items['returned_at'].isna())].copy()
completed_items['order_date'] = completed_items['created_at'].dt.tz_convert(None)
completed_items['month'] = month_start(completed_items['order_date'])

completed_orders = completed_items.groupby(['user_id', 'order_id'], as_index=False).agg(
    order_date=('order_date', 'min'),
    order_value=('sale_price', 'sum'),
)
completed_orders['month'] = month_start(completed_orders['order_date'])

dataset_max_date = completed_orders['order_date'].max().normalize()
analysis_start = pd.Timestamp('2020-01-01')
last_full_month_start = (dataset_max_date.to_period('M') - 1).to_timestamp()
analysis_cutoff = last_full_month_start + pd.offsets.MonthEnd(0)

sales_metrics = completed_items.groupby('month', as_index=False).agg(
    revenue=('sale_price', 'sum'),
    orders=('order_id', 'nunique'),
    units=('id', 'count'),
)
sales_metrics['aov'] = sales_metrics['revenue'] / sales_metrics['orders']

event_users = (
    events.dropna(subset=['user_id'])
    .assign(event_date=lambda df: df['created_at'].dt.tz_convert(None))
    .assign(month=lambda df: month_start(df['event_date']))
)
event_users = event_users[event_users['event_type'].isin(['department', 'purchase'])]

funnel_metrics = (
    event_users.groupby(['month', 'event_type'])['user_id']
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={'department': 'top_funnel_users', 'purchase': 'converted_users'})
)

monthly_product_metrics = sales_metrics.merge(funnel_metrics, on='month', how='left')
monthly_product_metrics['monthly_conversion_rate'] = (
    monthly_product_metrics['converted_users'] / monthly_product_metrics['top_funnel_users']
)

first_purchase = completed_orders.groupby('user_id')['month'].min().rename('first_purchase_month')
monthly_user_sales = (
    completed_orders.groupby(['user_id', 'month'], as_index=False)
    .agg(user_revenue=('order_value', 'sum'))
    .merge(first_purchase, on='user_id', how='left')
)

new_revenue = (
    monthly_user_sales.loc[monthly_user_sales['month'].eq(monthly_user_sales['first_purchase_month'])]
    .groupby('month')['user_revenue']
    .sum()
    .rename('New revenue')
)
returning_revenue = (
    monthly_user_sales.loc[monthly_user_sales['month'].gt(monthly_user_sales['first_purchase_month'])]
    .groupby('month')['user_revenue']
    .sum()
    .rename('Returning revenue')
)

monthly_mix = (
    pd.concat([new_revenue, returning_revenue], axis=1)
    .fillna(0)
    .reset_index()
)
monthly_mix['total_revenue'] = monthly_mix['New revenue'] + monthly_mix['Returning revenue']
monthly_mix['returning_share'] = monthly_mix['Returning revenue'] / monthly_mix['total_revenue']
monthly_mix = monthly_mix.merge(
    monthly_user_sales.groupby('month')['user_id'].nunique().rename('active_customers'),
    on='month',
    how='left',
)
monthly_mix = monthly_mix.merge(
    monthly_user_sales.loc[monthly_user_sales['month'].eq(monthly_user_sales['first_purchase_month'])]
    .groupby('month')['user_id']
    .nunique()
    .rename('new_customers'),
    on='month',
    how='left',
)
monthly_mix = monthly_mix.merge(
    monthly_user_sales.loc[monthly_user_sales['month'].gt(monthly_user_sales['first_purchase_month'])]
    .groupby('month')['user_id']
    .nunique()
    .rename('returning_customers'),
    on='month',
    how='left',
).fillna({'new_customers': 0, 'returning_customers': 0})

user_active_months = (
    completed_orders.groupby(['user_id', 'month'], as_index=False)
    .agg(last_order_date_in_month=('order_date', 'max'))
)
user_active_months = user_active_months[
    user_active_months['last_order_date_in_month'] <= dataset_max_date - pd.Timedelta(days=90)
].copy()

order_timestamps = completed_orders[['user_id', 'order_date']].drop_duplicates().sort_values(['user_id', 'order_date'])
order_timestamps['next_order_date'] = order_timestamps.groupby('user_id')['order_date'].shift(-1)

user_future_activity = user_active_months.merge(
    order_timestamps,
    left_on=['user_id', 'last_order_date_in_month'],
    right_on=['user_id', 'order_date'],
    how='left',
)
user_future_activity['has_future_order_90d'] = (
    user_future_activity['next_order_date'].notna()
    & ((user_future_activity['next_order_date'] - user_future_activity['last_order_date_in_month']).dt.days <= 90)
)

monthly_churn = user_future_activity.groupby('month', as_index=False).agg(
    active_customers=('user_id', 'nunique'),
    churned_customers_90d=('has_future_order_90d', lambda s: (~s).sum()),
)
monthly_churn['churn_rate_90d'] = monthly_churn['churned_customers_90d'] / monthly_churn['active_customers']

customer_snapshot = completed_orders[completed_orders['order_date'] <= analysis_cutoff].groupby('user_id', as_index=False).agg(
    first_purchase_date=('order_date', 'min'),
    last_purchase_date=('order_date', 'max'),
    lifetime_orders=('order_id', 'nunique'),
    lifetime_revenue=('order_value', 'sum'),
)
customer_snapshot['recency_days'] = (analysis_cutoff - customer_snapshot['last_purchase_date']).dt.days
high_value_threshold = customer_snapshot['lifetime_revenue'].quantile(0.75)


def assign_segment(row: pd.Series) -> str:
    if row['recency_days'] <= 90 and row['lifetime_orders'] >= 3 and row['lifetime_revenue'] >= high_value_threshold:
        return 'Champions'
    if row['recency_days'] <= 90 and row['lifetime_orders'] >= 2:
        return 'Active repeat'
    if row['recency_days'] <= 90:
        return 'Newly acquired'
    if row['recency_days'] <= 180:
        return 'At risk'
    return 'Dormant'


customer_snapshot['segment'] = customer_snapshot.apply(assign_segment, axis=1)
segment_order = ['Dormant', 'At risk', 'Newly acquired', 'Active repeat', 'Champions']
segment_summary = customer_snapshot.groupby('segment', as_index=False).agg(
    customers=('user_id', 'nunique'),
    revenue=('lifetime_revenue', 'sum'),
    avg_orders=('lifetime_orders', 'mean'),
)
segment_summary['customer_share'] = segment_summary['customers'] / segment_summary['customers'].sum()
segment_summary['revenue_share'] = segment_summary['revenue'] / segment_summary['revenue'].sum()
segment_summary['segment'] = pd.Categorical(segment_summary['segment'], categories=segment_order, ordered=True)
segment_summary = segment_summary.sort_values('segment')

shipping_window = completed_orders[
    (completed_orders['order_date'] >= pd.Timestamp('2021-12-16'))
    & (completed_orders['order_date'] <= pd.Timestamp('2022-02-14'))
].copy()
shipping_window['period'] = np.where(
    shipping_window['order_date'] < pd.Timestamp('2022-01-15'),
    'Pre-launch',
    'Post-launch',
)
shipping_window['free_shipping_eligible'] = shipping_window['order_value'] >= 100
shipping_baseline = shipping_window.groupby('period', as_index=False).agg(
    completed_orders=('order_id', 'nunique'),
    avg_order_value=('order_value', 'mean'),
    pct_orders_above_100=('free_shipping_eligible', 'mean'),
)

analysis_context = pd.DataFrame(
    {
        'metric': [
            'Latest completed-order date in dataset',
            'Last full month used for mix and segmentation',
            'Last mature month available for 90-day churn',
            'Purchasing customers in segment snapshot',
        ],
        'value': [
            dataset_max_date.date().isoformat(),
            analysis_cutoff.date().isoformat(),
            monthly_churn['month'].max().date().isoformat(),
            f"{customer_snapshot['user_id'].nunique():,}",
        ],
    }
)

analysis_context


,metric,value
0,Latest completed-order date in dataset,2026-04-10
1,Last full month used for mix and segmentation,2026-03-31
2,Last mature month available for 90-day churn,2026-01-01
3,Purchasing customers in segment snapshot,"26,513"


## 1. New vs Returning Revenue Mix by Month

This reproduces Task B and keeps both the monthly revenue stack and the returning-revenue share in one view. I exclude the current partial month so the trend is not distorted by incomplete April 2026 data.


In [4]:
mix_chart = monthly_mix[
    (monthly_mix['month'] >= analysis_start) & (monthly_mix['month'] <= last_full_month_start)
].copy()

fig_mix = make_subplots(specs=[[{'secondary_y': True}]])
fig_mix.add_trace(
    go.Bar(
        x=mix_chart['month'],
        y=mix_chart['New revenue'],
        name='New revenue',
        marker_color=PALETTE['gold'],
        hovertemplate='%{x|%b %Y}<br>New revenue: $%{y:,.0f}<extra></extra>',
    )
)
fig_mix.add_trace(
    go.Bar(
        x=mix_chart['month'],
        y=mix_chart['Returning revenue'],
        name='Returning revenue',
        marker_color=PALETTE['teal'],
        hovertemplate='%{x|%b %Y}<br>Returning revenue: $%{y:,.0f}<extra></extra>',
    )
)
fig_mix.add_trace(
    go.Scatter(
        x=mix_chart['month'],
        y=mix_chart['returning_share'] * 100,
        name='Returning share',
        mode='lines',
        line=dict(color=PALETTE['ink'], width=3),
        hovertemplate='%{x|%b %Y}<br>Returning share: %{y:.1f}%<extra></extra>',
    ),
    secondary_y=True,
)

style_figure(fig_mix, 'Monthly revenue mix: acquisition still drives most revenue')
fig_mix.update_layout(barmode='stack')
fig_mix.update_yaxes(title_text='Revenue ($)', tickprefix='$', secondary_y=False)
fig_mix.update_yaxes(title_text='Returning revenue share (%)', range=[0, 25], secondary_y=True)
fig_mix.show()


Takeaways:
- Returning revenue share rose from about 3.4% in the 2020 monthly average to 16.1% across the latest 12 full months.
- Even in the strongest months, returning customers still contribute less than 20% of revenue, so growth remains acquisition-led.
- The main retention opportunity is moving first-time buyers into a second purchase inside the first 90 days.


## 2. Monthly Churn Rate vs Revenue

This follows Task C. The 90-day churn definition is intentionally strict, so the churn axis is zoomed to show month-over-month movement rather than hiding it against a 0 to 100% scale.


In [5]:
churn_chart = monthly_churn.merge(sales_metrics[['month', 'revenue']], on='month', how='left')
churn_chart = churn_chart[churn_chart['month'] >= analysis_start].copy()
churn_axis_floor = max(90, (churn_chart['churn_rate_90d'].min() * 100) - 1.5)

fig_churn = make_subplots(specs=[[{'secondary_y': True}]])
fig_churn.add_trace(
    go.Bar(
        x=churn_chart['month'],
        y=churn_chart['revenue'],
        name='Revenue',
        marker_color=PALETTE['blue'],
        opacity=0.85,
        hovertemplate='%{x|%b %Y}<br>Revenue: $%{y:,.0f}<extra></extra>',
    )
)
fig_churn.add_trace(
    go.Scatter(
        x=churn_chart['month'],
        y=churn_chart['churn_rate_90d'] * 100,
        name='90-day churn rate',
        mode='lines+markers',
        marker=dict(size=6, color=PALETTE['coral']),
        line=dict(color=PALETTE['coral'], width=3),
        hovertemplate='%{x|%b %Y}<br>90-day churn: %{y:.1f}%<extra></extra>',
    ),
    secondary_y=True,
)

style_figure(fig_churn, 'Revenue rises when churn pressure eases')
fig_churn.update_yaxes(title_text='Revenue ($)', tickprefix='$', secondary_y=False)
fig_churn.update_yaxes(title_text='90-day churn rate (%)', range=[churn_axis_floor, 100.5], secondary_y=True)
fig_churn.show()


Takeaways:
- The 90-day churn definition from Part 1 produces very high monthly values, averaging about 98.6% since 2020 and bottoming out near 93.1% in January 2026.
- Revenue and churn move in opposite directions, with a roughly -0.78 correlation after 2020.
- In plain terms, this dataset behaves like a business with strong top-of-funnel acquisition but very weak repeat-purchase frequency.


## 3. Customer Segment Split

I used a simple RFM-style segmentation at the March 31, 2026 snapshot date:
- `Champions`: purchased in the last 90 days, at least 3 orders, and above the 75th percentile for lifetime revenue
- `Active repeat`: purchased in the last 90 days with at least 2 orders
- `Newly acquired`: first and only purchase inside the last 90 days
- `At risk`: last purchase 91 to 180 days ago
- `Dormant`: last purchase more than 180 days ago


In [6]:
fig_segments = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type': 'domain'}, {'type': 'xy'}]],
    subplot_titles=('Share of purchasing customers', 'Share of lifetime revenue'),
)

fig_segments.add_trace(
    go.Pie(
        labels=segment_summary['segment'].astype(str),
        values=segment_summary['customers'],
        hole=0.58,
        sort=False,
        marker=dict(colors=[SEGMENT_COLORS[label] for label in segment_summary['segment'].astype(str)]),
        textinfo='label+percent',
        hovertemplate='%{label}<br>Customers: %{value:,}<br>Share: %{percent}<extra></extra>',
    ),
    row=1,
    col=1,
)

fig_segments.add_trace(
    go.Bar(
        y=segment_summary['segment'].astype(str),
        x=segment_summary['revenue_share'] * 100,
        orientation='h',
        marker_color=[SEGMENT_COLORS[label] for label in segment_summary['segment'].astype(str)],
        text=(segment_summary['revenue_share'] * 100).round(1).astype(str) + '%',
        textposition='outside',
        hovertemplate='%{y}<br>Revenue share: %{x:.1f}%<br>Avg orders: %{customdata:.2f}<extra></extra>',
        customdata=segment_summary['avg_orders'],
        showlegend=False,
    ),
    row=1,
    col=2,
)

style_figure(fig_segments, 'Most purchasers are dormant, but active repeat buyers are economically denser', height=540)
fig_segments.update_xaxes(title_text='Share of lifetime revenue (%)', row=1, col=2)
fig_segments.update_yaxes(title_text='', row=1, col=2)
fig_segments.show()


Takeaways:
- As of March 31, 2026, dormant customers represent 73.7% of purchasers and 72.5% of lifetime revenue.
- `Active repeat` plus `Champions` are only about 2.6% of customers but almost 4.9% of revenue, which makes them disproportionately valuable.
- This split is intentionally simple and actionable: recency isolates reactivation risk, while frequency plus spend isolates the small high-value base worth protecting.


## Appendix: Experiment Design for Free Shipping over $100

Task D in `PART_1` used a pre/post proxy. For a real decision, the cleaner approach is a randomized experiment.

| Element | Recommendation |
| --- | --- |
| Hypothesis | Free shipping above $100 increases completed-order conversion and nudges more carts across the threshold. |
| Randomization unit | `user_id` for signed-in users, cookie or session for anonymous visitors. |
| Control | Current shipping policy. |
| Treatment | Free shipping once cart value is at least $100. |
| Primary metric | Completed-order conversion rate. |
| Secondary metrics | Average order value, revenue per visitor, share of orders above $100. |
| Guardrails | Gross margin per order, return rate, cancellation rate, delivery cost per order. |
| Key cuts | New vs returning customers, traffic source, geography, carts already near the $100 threshold. |
| Decision rule | Only ship if conversion lift is positive and margin guardrails stay inside tolerance. |

The local launch window is directionally interesting but not causal: pre-launch AOV was about $75.10 with 20.0% of orders above $100, while post-launch AOV was about $74.48 with 26.6% above $100. That makes the threshold test worth running, but only a randomized design can tell whether the policy truly caused the shift.


In [7]:
shipping_baseline.assign(
    avg_order_value=lambda df: df['avg_order_value'].round(2),
    pct_orders_above_100=lambda df: (df['pct_orders_above_100'] * 100).round(1),
)


,period,completed_orders,avg_order_value,pct_orders_above_100
0,Post-launch,192,74.48,26.60
1,Pre-launch,170,75.10,20.00


## Final Readout

The dataset tells a consistent story: revenue growth is mostly acquisition-led, returning revenue is improving but still small, and the 90-day churn definition highlights how rarely customers come back quickly. The most useful business actions from this notebook are:
- build a second-purchase retention program for the first 90 days after acquisition,
- protect the small repeat-buyer base with targeted CRM and remarketing,
- test the free-shipping threshold with a randomized design instead of relying on a pre/post comparison.
